In [15]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")

APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN")

URL = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"

headers = {
    "X-App-Token": APP_TOKEN
}

columns = [
    "id",
    "date",
    "primary_type",
    "description",
    "location_description",
    "arrest",
    "domestic",
    "beat",
    "district",
    "ward",
    "community_area",
    "year",
    "latitude",
    "longitude"
]

In [19]:
params = {
    "$select": ",".join(columns),
    "$where": "year >= 2021 AND year <= 2026",
    "$order": "id",
    "$limit": 100
}

response = requests.get(
    URL,
    headers=headers,
    params=params,
    timeout=60
)

response.raise_for_status()

test_data = response.json()
test_df = pd.DataFrame(test_data)

print(test_df.shape)
print(test_df["year"].value_counts().sort_index())

(100, 14)
year
2021    100
Name: count, dtype: int64


In [20]:
BATCH_SIZE = 50000
offset = 0

all_batches = []

while True:

    params = {
        "$select": ",".join(columns),
        "$where": "year >= 2021 AND year <= 2026",
        "$order": "id",
        "$limit": BATCH_SIZE,
        "$offset": offset
    }

    response = requests.get(
        URL,
        headers=headers,
        params=params,
        timeout=120
    )

    response.raise_for_status()

    batch = response.json()

    if len(batch) == 0:
        break

    batch_df = pd.DataFrame(batch)
    all_batches.append(batch_df)

    print(f"Downloaded: {offset + len(batch):,} rows")

    offset += BATCH_SIZE

Downloaded: 50,000 rows
Downloaded: 100,000 rows
Downloaded: 150,000 rows
Downloaded: 200,000 rows
Downloaded: 250,000 rows
Downloaded: 300,000 rows
Downloaded: 350,000 rows
Downloaded: 400,000 rows
Downloaded: 450,000 rows
Downloaded: 500,000 rows
Downloaded: 550,000 rows
Downloaded: 600,000 rows
Downloaded: 650,000 rows
Downloaded: 700,000 rows
Downloaded: 750,000 rows
Downloaded: 800,000 rows
Downloaded: 850,000 rows
Downloaded: 900,000 rows
Downloaded: 950,000 rows
Downloaded: 1,000,000 rows
Downloaded: 1,050,000 rows
Downloaded: 1,100,000 rows
Downloaded: 1,150,000 rows
Downloaded: 1,200,000 rows
Downloaded: 1,250,000 rows
Downloaded: 1,300,000 rows
Downloaded: 1,350,000 rows
Downloaded: 1,388,164 rows


In [22]:
crime_df = pd.concat(
    all_batches,
    ignore_index=True
)

crime_df.shape
crime_df["year"].value_counts().sort_index()

year
2021    210009
2022    240501
2023    263852
2024    259655
2025    238133
2026    176014
Name: count, dtype: int64

In [23]:
print("Shape:", crime_df.shape)

print("\nYears:")
print(crime_df["year"].value_counts().sort_index())

print("\nDuplicates:")
print(crime_df["id"].duplicated().sum())

print("\nMissing coordinates:")
print(crime_df[["latitude", "longitude"]].isnull().sum())

Shape: (1388164, 14)

Years:
year
2021    210009
2022    240501
2023    263852
2024    259655
2025    238133
2026    176014
Name: count, dtype: int64

Duplicates:
0

Missing coordinates:
latitude     20507
longitude    20507
dtype: int64


In [24]:
crime_df.to_csv(
    "../data/raw_chicago_crimes.csv",
    index=False
)

In [25]:
import os

size_mb = os.path.getsize(
    "../data/raw_chicago_crimes.csv"
) / (1024 ** 2)

print(f"File size: {size_mb:.2f} MB")

File size: 178.05 MB
